# P3 — Global LightGBM Quantile Forecasting

Huấn luyện LightGBM global trên mẫu 3.474 chuỗi, 3 rolling windows và 9 mức quantile.

- Input: `data/processed/data_long.parquet`, `common/{config.yaml,rolling_windows.csv,sample_ids.csv}`
- Output chính: `data/outputs/forecasts/lgbm.parquet`
- Output hỗ trợ: validation forecast, feature importance, runtime và run config trong `data/outputs/p3/`
- Calibration chung được thực hiện ở P5.

In [1]:
# Thiết lập môi trường và cấu hình chung
from pathlib import Path
import gc, json, platform, random, time, warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
CWD = Path.cwd()
ROOT = CWD if (CWD / "common/config.yaml").exists() else CWD.parent
assert (ROOT / "common/config.yaml").exists(), f"Không xác định được project root từ {CWD}"

with open(ROOT / "common/config.yaml", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

SEED = int(CFG["seed"]); H = int(CFG["horizon"])
Q = sorted(float(q) for q in CFG["quantiles"])
random.seed(SEED); np.random.seed(SEED)

PATH_LONG = ROOT / "data/processed/data_long.parquet"
PATH_WINDOWS = ROOT / "common/rolling_windows.csv"
PATH_SAMPLE = ROOT / "common/sample_ids.csv"
FORECAST_DIR = ROOT / "data/outputs/forecasts"
P3_OUT = ROOT / "data/outputs/p3"

for name, path in {"data_long": PATH_LONG, "rolling_windows": PATH_WINDOWS, "sample_ids": PATH_SAMPLE}.items():
    assert path.exists(), f"Thiếu {name}: {path}"

print("Python:", platform.python_version(), "| LightGBM:", lgb.__version__)
print("Project root:", ROOT)

Python: 3.13.11 | LightGBM: 4.7.0
Project root: c:\Users\Dell\m5-intermittent-forecasting


In [2]:
# Nạp đúng các cột P3 cần dùng
windows = pd.read_csv(PATH_WINDOWS)
sample_ids = pd.read_csv(PATH_SAMPLE)["series_id"].astype(str)

INPUT_COLS = ["series_id", "date", "d_int", "sales", "wday", "month",
              "is_event", "snap", "sell_price", "segment"]
data = pd.read_parquet(PATH_LONG, columns=INPUT_COLS)
data["date"] = pd.to_datetime(data["date"])
data["series_id"] = data["series_id"].astype("category")
data["segment"] = data["segment"].astype("category")
data["d_int"] = pd.to_numeric(data["d_int"]).astype("int32")
data["sales"] = pd.to_numeric(data["sales"]).astype("float32")
data = data.sort_values(["series_id", "d_int"]).reset_index(drop=True)

print("data_long:", data.shape, "| sample ids:", len(sample_ids), "| windows:", len(windows))
display(windows)

data_long: (6743034, 10) | sample ids: 3474 | windows: 3


,window,train_end,val_end,test_start,test_end
0,W1,1885,1913,1914,1941
1,W2,1857,1885,1886,1913
2,W3,1829,1857,1858,1885


In [3]:
# Kiểm tra input contract
expected_ids = set(sample_ids)
actual_ids = set(data["series_id"].astype(str).unique())
assert actual_ids == expected_ids, "data_long không khớp common/sample_ids.csv"
assert data.duplicated(["series_id", "d_int"]).sum() == 0
assert data["d_int"].min() == 1 and data["d_int"].max() == int(CFG["split"]["test_end"])
assert data.groupby("series_id", observed=True).size().nunique() == 1
assert data.groupby("d_int")["date"].nunique().max() == 1
assert len(windows) >= 3 and len(Q) == 9

N_SERIES = len(actual_ids)
segment_counts = data[["series_id", "segment"]].drop_duplicates()["segment"].value_counts()
print(f"Input hợp lệ: {N_SERIES:,} chuỗi × {data['d_int'].nunique():,} ngày")
display(segment_counts)

Input hợp lệ: 3,474 chuỗi × 1,941 ngày


segment
intermittent    1000
lumpy           1000
smooth           980
erratic          494
Name: count, dtype: int64

## Feature engineering

Các feature từ sales đều lùi ít nhất 28 ngày để toàn bộ horizon được tính từ thông tin đã biết; calendar, SNAP, event và giá là biến ngoại sinh.

In [4]:
# Tạo lag, rolling, days-since-last-sale và biến ngoại sinh
t_feature = time.perf_counter()
sales_group = data.groupby("series_id", observed=True, sort=False)["sales"]
for lag in (28, 35, 56):
    data[f"lag_{lag}"] = sales_group.shift(lag).astype("float32")

shifted_28 = sales_group.shift(H)
shifted_group = shifted_28.groupby(data["series_id"], observed=True, sort=False)
data["rmean_7"] = shifted_group.transform(lambda s: s.rolling(7, min_periods=2).mean()).astype("float32")
data["rmean_28"] = shifted_group.transform(lambda s: s.rolling(28, min_periods=5).mean()).astype("float32")
data["rstd_28"] = shifted_group.transform(lambda s: s.rolling(28, min_periods=5).std()).astype("float32")

last_sale_day = data["d_int"].where(data["sales"].gt(0)).groupby(data["series_id"], observed=True).ffill()
days_since_sale = (data["d_int"] - last_sale_day).fillna(data["d_int"]).astype("float32")
data["dsls_28"] = days_since_sale.groupby(data["series_id"], observed=True).shift(H).astype("float32")

data["price_missing"] = data["sell_price"].isna().astype("int8")
data["price_chg"] = (data.groupby("series_id", observed=True, sort=False)["sell_price"]
                           .pct_change(fill_method=None).replace([np.inf, -np.inf], 0).fillna(0).astype("float32"))
data["sell_price"] = data["sell_price"].fillna(0).astype("float32")
for c in ["wday", "month", "is_event", "snap"]:
    data[c] = data[c].fillna(0).astype("int8")

FEATURE_COLS = ["lag_28", "lag_35", "lag_56", "rmean_7", "rmean_28", "rstd_28",
                "dsls_28", "wday", "month", "is_event", "snap",
                "sell_price", "price_chg", "price_missing"]
CAT_COLS = ["wday", "month"]
print(f"Đã tạo {len(FEATURE_COLS)} features trong {time.perf_counter()-t_feature:.1f}s")

Đã tạo 14 features trong 2.6s


In [5]:
# Audit feature và cấu hình LightGBM
usable = data["lag_56"].notna()
nulls = data.loc[usable, FEATURE_COLS].isna().sum()
assert nulls.sum() == 0, f"Feature còn NA:\n{nulls[nulls > 0]}"

for _, w in windows.iterrows():
    tr, ve, ts, te = map(int, [w.train_end, w.val_end, w.test_start, w.test_end])
    assert ve - tr == H and te - ts + 1 == H and ts == ve + 1

BASE_PARAMS = dict(
    objective="quantile", metric="quantile", learning_rate=0.05,
    num_leaves=63, min_data_in_leaf=100, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, verbosity=-1, num_threads=4,
    seed=SEED, feature_fraction_seed=SEED, bagging_seed=SEED,
)
MAX_ROUNDS = 300
EARLY_STOPPING_ROUNDS = 30
FORECAST_COLS = ["series_id", "date", "d_int", "quantile_level", "value", "model"]
print("Feature audit đạt | 3 windows × 9 quantiles = 27 models")

Feature audit đạt | 3 windows × 9 quantiles = 27 models


## Rolling-origin training

Mỗi window dùng train để học, validation để early stopping và lưu forecast phục vụ calibration, test để đánh giá. Quantile crossing được xử lý bằng monotone rearrangement.

In [ ]:
# Hàm huấn luyện một rolling window và check forecast cuối
def to_long_forecast(base, prediction_matrix, model_name, window_name=None):
    prediction_matrix = np.sort(np.clip(prediction_matrix, 0, None), axis=1)
    out = base.loc[base.index.repeat(len(Q))].reset_index(drop=True).copy()
    out["series_id"] = out["series_id"].astype(str)
    out["quantile_level"] = np.tile(Q, len(base))
    out["value"] = prediction_matrix.reshape(-1).astype("float32")
    out["model"] = model_name
    if window_name is not None:
        out["window"] = window_name
    return out

def train_window(w):
    window_name = str(w["window"])
    train_end, val_end = int(w["train_end"]), int(w["val_end"])
    test_start, test_end = int(w["test_start"]), int(w["test_end"])
    train_mask = (data["d_int"] <= train_end) & data["lag_56"].notna()
    val_mask = data["d_int"].between(train_end + 1, val_end)
    test_mask = data["d_int"].between(test_start, test_end)

    X_train, y_train = data.loc[train_mask, FEATURE_COLS], data.loc[train_mask, "sales"]
    X_val, y_val = data.loc[val_mask, FEATURE_COLS], data.loc[val_mask, "sales"]
    X_test = data.loc[test_mask, FEATURE_COLS]
    val_base = data.loc[val_mask, ["series_id", "date", "d_int"]].reset_index(drop=True)
    test_base = data.loc[test_mask, ["series_id", "date", "d_int"]].reset_index(drop=True)
    assert len(val_base) == N_SERIES * H and len(test_base) == N_SERIES * H

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_COLS, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_COLS, reference=dtrain, free_raw_data=False)
    val_preds, test_preds, runtime_rows, importance_rows = [], [], [], []
    print(f"{window_name}: train={len(X_train):,} | val={len(X_val):,} | test={len(X_test):,}")

    for q in Q:
        t0 = time.perf_counter()
        model = lgb.train(
            {**BASE_PARAMS, "alpha": q}, dtrain, num_boost_round=MAX_ROUNDS,
            valid_sets=[dval], valid_names=["validation"],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(0)],
        )
        best_iteration = int(model.best_iteration or MAX_ROUNDS)
        val_preds.append(np.clip(model.predict(X_val, num_iteration=best_iteration), 0, None))
        test_preds.append(np.clip(model.predict(X_test, num_iteration=best_iteration), 0, None))
        seconds = time.perf_counter() - t0
        runtime_rows.append({"window": window_name, "quantile_level": q,
                             "best_iteration": best_iteration, "seconds": seconds})
        if np.isclose(q, 0.5):
            importance_rows.extend({"window": window_name, "feature": feature, "gain": float(gain)}
                                   for feature, gain in zip(FEATURE_COLS, model.feature_importance("gain")))
        print(f"  q={q:>5}: iter={best_iteration:>3} | {seconds:6.1f}s")

    val_fc = to_long_forecast(val_base, np.column_stack(val_preds), "lgbm", window_name)
    test_fc = to_long_forecast(test_base, np.column_stack(test_preds), "lgbm")
    return (test_fc[FORECAST_COLS], val_fc[FORECAST_COLS + ["window"]],
            pd.DataFrame(runtime_rows), pd.DataFrame(importance_rows))

def validate_forecast(df):
    expected_rows = len(windows) * N_SERIES * H * len(Q)
    assert list(df.columns) == FORECAST_COLS
    assert len(df) == expected_rows, f"{len(df):,} != {expected_rows:,}"
    assert df["series_id"].nunique() == N_SERIES
    assert set(np.round(df["quantile_level"].unique(), 6)) == set(np.round(Q, 6))
    assert df["value"].notna().all() and np.isfinite(df["value"]).all() and (df["value"] >= 0).all()
    assert set(df["model"].unique()) == {"lgbm"}
    assert df.duplicated(["series_id", "d_int", "quantile_level"]).sum() == 0
    wide = df.pivot(index=["series_id", "d_int"], columns="quantile_level", values="value")[Q]
    assert not (np.diff(wide.to_numpy(), axis=1) < -1e-8).any(), "Còn quantile crossing"
    print(f"Forecast hợp lệ: {len(df):,} dòng | {N_SERIES:,} chuỗi | không crossing")

In [ ]:
# Huấn luyện 9 quantile trên 3 rolling window
test_parts, val_parts, runtime_parts, importance_parts = [], [], [], []
for _, window in windows.iterrows():
    test_fc, val_fc, runtime_df, importance_df = train_window(window)
    test_parts.append(test_fc); val_parts.append(val_fc)
    runtime_parts.append(runtime_df); importance_parts.append(importance_df)
    gc.collect()

fc = pd.concat(test_parts, ignore_index=True)
validation_fc = pd.concat(val_parts, ignore_index=True)
runtime = pd.concat(runtime_parts, ignore_index=True)
feature_importance = pd.concat(importance_parts, ignore_index=True)

W1: train=6,353,946 | val=97,272 | test=97,272
  q=0.005: iter=  1 |    5.0s
  q=0.025: iter= 79 |   15.7s
  q=0.165: iter=300 |   39.3s
  q= 0.25: iter=300 |   40.7s
  q=  0.5: iter=180 |   28.7s
  q= 0.75: iter=254 |   35.1s
  q=0.835: iter=296 |   37.2s
  q=0.975: iter=300 |   37.0s
  q=0.995: iter=300 |   33.3s
W2: train=6,256,674 | val=97,272 | test=97,272
  q=0.005: iter=  1 |    5.6s
  q=0.025: iter=101 |   18.9s
  q=0.165: iter=166 |   27.2s
  q= 0.25: iter=299 |   41.7s
  q=  0.5: iter=193 |   31.5s
  q= 0.75: iter=163 |   26.2s
  q=0.835: iter=297 |   56.8s
  q=0.975: iter=166 |   42.1s
  q=0.995: iter=156 |   37.1s
W3: train=6,159,402 | val=97,272 | test=97,272
  q=0.005: iter=  1 |   10.0s
  q=0.025: iter= 65 |   23.4s
  q=0.165: iter=260 |   65.1s
  q= 0.25: iter=289 |   66.4s
  q=  0.5: iter=209 |   53.9s
  q= 0.75: iter=300 |   61.8s
  q=0.835: iter=279 |   59.6s
  q=0.975: iter=300 |   57.5s
  q=0.995: iter=150 |   35.0s


In [ ]:
# Check kết quả và ghi file
validate_forecast(fc)
assert len(validation_fc) == len(windows) * N_SERIES * H * len(Q)
assert validation_fc.duplicated(["window", "series_id", "d_int", "quantile_level"]).sum() == 0

feature_importance["gain_share"] = feature_importance.groupby("window")["gain"].transform(
    lambda s: s / s.sum() if s.sum() else 0
)
runtime_summary = runtime.groupby("window", as_index=False).agg(
    seconds=("seconds", "sum"), mean_best_iteration=("best_iteration", "mean")
)
display(runtime_summary)
display(feature_importance.groupby("feature", as_index=False)["gain_share"].mean()
        .sort_values("gain_share", ascending=False).head(10))

Forecast hợp lệ: 2,626,344 dòng | 3,474 chuỗi | không crossing


,window,seconds,mean_best_iteration
0,W1,272.023561,223.333333
1,W2,287.062757,171.333333
2,W3,432.794406,205.888889


,feature,gain_share
8,rmean_28,0.673617
10,rstd_28,0.169007
11,sell_price,0.062598
9,rmean_7,0.061320
7,price_missing,0.010908
0,dsls_28,0.009472
13,wday,0.003540
5,month,0.002523
4,lag_56,0.002489
2,lag_28,0.002421


In [ ]:
# Lưu output và artifact
FORECAST_DIR.mkdir(parents=True, exist_ok=True)
P3_OUT.mkdir(parents=True, exist_ok=True)
fc.to_parquet(FORECAST_DIR / "lgbm.parquet", index=False)
validation_fc.to_parquet(P3_OUT / "lgbm_validation.parquet", index=False)
feature_importance.to_parquet(P3_OUT / "lgbm_feature_importance.parquet", index=False)
runtime.to_csv(P3_OUT / "lgbm_runtime.csv", index=False)

run_config = {
    "python": platform.python_version(), "lightgbm": lgb.__version__, "seed": SEED,
    "horizon": H, "quantiles": Q, "n_series": N_SERIES,
    "n_windows": len(windows), "features": FEATURE_COLS,
    "categorical_features": CAT_COLS, "max_rounds": MAX_ROUNDS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS, "params": BASE_PARAMS,
}
with open(P3_OUT / "lgbm_run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print("Đã lưu:", FORECAST_DIR / "lgbm.parquet")
print("Forecast rows:", f"{len(fc):,}", "| runtime:", f"{runtime['seconds'].sum()/60:.1f} phút")

Đã lưu: c:\Users\Dell\m5-intermittent-forecasting\data\outputs\forecasts\lgbm.parquet
Forecast rows: 2,626,344 | runtime: 16.5 phút
